In [1]:
from ultralytics import YOLO

# 1️⃣ Model load
model = YOLO(r"D:\saylani class\Deep learning\best.pt")

# 2️⃣ Image ka sahi path
image_list_path =[r"D:\data\archive (4)\Pakistani License Number Plates Data\Cars\DSC_1056.jpg",
                 r"D:\data\archive (4)\Pakistani License Number Plates Data\Cars\DSC_1090.jpg"] 

# 3️⃣ Prediction
results = model.predict(source=image_list_path, show=True, save=True)

# 4️⃣ Output
print(results)



0: 640x640 1 objects, 240.2ms
1: 640x640 1 objects, 240.2ms
Speed: 11.4ms preprocess, 240.2ms inference, 4.7ms postprocess per image at shape (1, 3, 640, 640)
Results saved to runs\detect\predict5
[ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'objects'}
obb: None
orig_img: array([[[142, 161, 176],
        [142, 161, 176],
        [140, 161, 176],
        ...,
        [ 88,  83,  82],
        [ 88,  83,  82],
        [ 90,  85,  84]],

       [[140, 159, 174],
        [140, 159, 174],
        [138, 159, 174],
        ...,
        [ 87,  82,  81],
        [ 86,  81,  80],
        [ 88,  83,  82]],

       [[139, 158, 171],
        [139, 158, 171],
        [139, 158, 171],
        ...,
        [ 90,  83,  80],
        [ 90,  83,  80],
        [ 92,  85,  82]],

       ...,

       [[ 70,  78,  85],
        [ 71,  79,  86],
        [ 71,  79,  86],
        ...,
        [115, 116, 120],
   

In [2]:
# Confidence print karna
for box in results[0].boxes:
    print("Label:", results[0].names[int(box.cls)])
    print("Confidence:", float(box.conf))


Label: objects
Confidence: 0.8616003394126892


In [3]:
# Check if a library is installed
libraries = ["ultralytics", "easyocr", "cv2", "pandas"]

for lib in libraries:
    try:
        __import__(lib)
        print(f"✅ {lib} is installed")
    except ImportError:
        print(f"❌ {lib} is NOT installed")


✅ ultralytics is installed
✅ easyocr is installed
✅ cv2 is installed
✅ pandas is installed


In [6]:
from ultralytics import YOLO
import cv2, torch, pandas as pd, easyocr, os

# ===== Settings =====
MODEL_PATH   = r"D:\saylani class\Deep learning\best.pt"          # آپ کا YOLO ماڈل
INPUT_VIDEO  = r"D:\pictures.mp4"        # آپ کی ویڈیو
OUTPUT_VIDEO = "output_annotated.mp4"  # annotated video save
CONF         = 0.25
IOU          = 0.45
DEVICE       = 0 if torch.cuda.is_available() else "cpu"
CSV_PATH     = "lp_detections.csv"

# ===== Load model & OCR reader =====
model = YOLO(MODEL_PATH)
reader = easyocr.Reader(['en'])  # English OCR

# ===== Video setup =====
cap = cv2.VideoCapture(INPUT_VIDEO)
w  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps= cap.get(cv2.CAP_PROP_FPS) or 30.0
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (w,h))

rows = []
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # YOLO prediction on frame
    res = model.predict(frame, conf=CONF, iou=IOU, device=DEVICE, verbose=False)[0]

    if res.boxes is not None and len(res.boxes) > 0:
        boxes = res.boxes.xyxy.cpu().numpy()
        scores= res.boxes.conf.cpu().numpy()
        for i, (x1,y1,x2,y2) in enumerate(boxes):
            conf_score = float(scores[i])
            # Crop plate area for OCR
            plate_crop = frame[int(y1):int(y2), int(x1):int(x2)]
            plate_text = ""
            if plate_crop.size > 0:
                result = reader.readtext(plate_crop)
                if result:
                    plate_text = result[0][1]  # OCR text

            # Draw bounding box + plate number
            label = f"{plate_text} {conf_score:.2f}" if plate_text else f"{conf_score:.2f}"
            cv2.rectangle(frame, (int(x1),int(y1)), (int(x2),int(y2)), (0,255,0), 2)
            cv2.putText(frame, label, (int(x1), max(0,int(y1)-5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

            # Log to CSV
            rows.append({
                "frame": frame_idx,
                "x1": float(x1), "y1": float(y1),
                "x2": float(x2), "y2": float(y2),
                "conf": conf_score,
                "plate_text": plate_text
            })

    out.write(frame)
    frame_idx += 1

cap.release()
out.release()

# ===== Save CSV =====
pd.DataFrame(rows).to_csv(CSV_PATH, index=False)
print(f"\n✅ Video saved: {OUTPUT_VIDEO}")
print(f"✅ Detections CSV saved: {CSV_PATH}")


Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.



✅ Video saved: output_annotated.mp4
✅ Detections CSV saved: lp_detections.csv
